# Pertemuan 12 — Praproses dan Pemodelan
## Proyek Akhir Klasifikasi: k-NN dan Naive Bayes

Pembelajaran Mesin INF62325 · S1 Informatika · UIN Siber Syekh Nurjati Cirebon

Nama: Muhammad Raffi
NIM: 2488010046

---
## Langkah 0 — Samakan versi scikit-learn

Jalankan dua sel berikut setiap kali membuka notebook ini, sebelum sel apa pun
yang lain. Berkas model menyimpan rujukan ke kelas internal scikit-learn versi
tertentu, sehingga versi pelatihan dan versi pemuatan harus sama di seluruh
pertemuan. Angka 1.9.1 berlaku November 2026.

In [ ]:
!pip install -q scikit-learn==1.9.1

**Wajib sekarang:** buka menu **Runtime**, pilih **Restart session**, lalu jalankan
ulang dari sel paling atas. Tanpa restart, Python masih memakai versi lama yang
sudah terlanjur dimuat ke memori.

In [ ]:
import sklearn, pandas as pd, numpy as np, joblib

print('scikit-learn :', sklearn.__version__)
print('pandas       :', pd.__version__)

assert sklearn.__version__ == '1.9.1', (
    'Versi belum sesuai. Jalankan sel pip di atas, lalu '
    'Runtime -> Restart session, lalu ulangi dari sel pertama.'
)
print('Versi sudah sesuai.')

---
## Langkah 1 — Memuat dan Membagi Data

Sel berikut memasang Drive, membaca berkas `data.csv` (UCI Higher Education Predict Dropout and Academic Success), dan membaginya menjadi data latih (80%) dan data uji (20%) secara berstrata.

In [ ]:
import glob, os, time
from google.colab import drive

drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/proyek_akhir_ml'
os.makedirs(FOLDER, exist_ok=True)

print('Mencari berkas data di Google Drive, mohon tunggu sebentar...')
temuan = sorted(glob.glob('/content/drive/MyDrive/**/data.csv', recursive=True))

if not temuan:
    raise FileNotFoundError(
        'Berkas data.csv tidak ditemukan di Google Drive Anda.'
    )

PATH = temuan[0]
if len(temuan) > 1:
    print(f'Ada {len(temuan)} salinan, yang dipakai salinan pertama:')
    for t in temuan:
        print('  -', t)

mentah = pd.read_csv(PATH, sep=';')
print('\nBerkas dipakai :', PATH)
print('Ukuran data    :', mentah.shape)

### Temuan EDA Pertemuan 11 yang diterapkan

1. **Ketidakseimbangan Kelas Target:** Distribusi kelas terdiri dari `Graduate` (49,93%), `Dropout` (32,12%), dan `Enrolled` (17,95%). Metrik utama menggunakan `f1_macro` dan pembagian data wajib memakai `stratify=y`.
2. **Integritas Data:** Tidak terdapat nilai kosong (*missing value*) maupun baris duplikat. Nilai 0 pada fitur nilai semester (`grade = 0`) dan usia dewasa (hingga 70 tahun) dipertahankan karena merupakan sinyal perilaku akademik yang valid.
3. **Seluruh Dataset Digunakan:** Ukuran dataset adalah 4.424 baris sehingga seluruh data diproses langsung (tanpa *subsampling*) dengan rasio 80% latih (3.539 baris) dan 20% uji (885 baris).

In [ ]:
TARGET = 'Target'
SEED = 42

df = mentah.copy()
print('Ukuran data siap bagi:', df.shape)
print('\nDistribusi kelas target:')
print(df[TARGET].value_counts(normalize=True).round(4))

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Pemisahan fitur kategorikal berkode angka dan fitur numerik kontinu
kategorik = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Daytime/evening attendance\t', 'Previous qualification', 'Nacionality',
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", 'Displaced', 'Educational special needs', 'Debtor',
    'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International'
]
kategorik = [c for c in kategorik if c in X.columns]
numerik = [c for c in X.columns if c not in kategorik]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print('Total fitur    :', X.shape[1], 'kolom')
print('Fitur numerik  :', len(numerik), 'kolom')
print('Fitur kategorik:', len(kategorik), 'kolom')
print('Latih          :', X_train.shape)
print('Uji            :', X_test.shape)

---
## Langkah 2 — Praproses di dalam Pipeline

Data dibagi terlebih dahulu, lalu transformasi dipasang hanya pada data latih menggunakan `ColumnTransformer` di dalam `Pipeline` agar bebas dari kebocoran data (*data leakage*) di setiap lipatan validasi silang.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def buat_preprocessor(pakai_skala: bool):
    """pakai_skala=True untuk model berbasis jarak seperti k-NN."""
    langkah_numerik = [('imputasi', SimpleImputer(strategy='median'))]
    if pakai_skala:
        langkah_numerik.append(('skala', StandardScaler()))

    alur_kategorik = Pipeline([
        ('imputasi', SimpleImputer(strategy='most_frequent')),
        ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    return ColumnTransformer([
        ('num', Pipeline(langkah_numerik), numerik),
        ('kat', alur_kategorik, kategorik),
    ])


contoh = buat_preprocessor(True).fit_transform(X_train)
print('Jumlah kolom setelah encoding:', contoh.shape[1])

---
## Langkah 3 — Melatih k-NN

Pencarian parameter dilakukan pada kombinasi jumlah tetangga (`n_neighbors`), pembobotan (`weights`), dan metrik jarak (`metric`) menggunakan **5-Fold Stratified Cross-Validation** dengan metrik `f1_macro`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

alur_knn = Pipeline([('prep', buat_preprocessor(pakai_skala=True)),
                     ('model', KNeighborsClassifier())])

ruang_knn = {
    'model__n_neighbors': list(range(1, 32, 2)),
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}

mulai = time.perf_counter()
cari_knn = GridSearchCV(alur_knn, ruang_knn, cv=cv, scoring='f1_macro', n_jobs=-1)
cari_knn.fit(X_train, y_train)
knn_terbaik = cari_knn.best_estimator_

print('Parameter terbaik :', cari_knn.best_params_)
print('Skor f1_macro CV  :', round(cari_knn.best_score_, 4))
print('Waktu pencarian   :', round(time.perf_counter() - mulai, 1), 'detik')

In [ ]:
import matplotlib.pyplot as plt

hasil = pd.DataFrame(cari_knn.cv_results_)
best_w = cari_knn.best_params_['model__weights']
best_m = cari_knn.best_params_['model__metric']

baris = hasil[(hasil['param_model__weights'] == best_w) &
              (hasil['param_model__metric'] == best_m)]

plt.figure(figsize=(7, 4))
plt.plot(baris['param_model__n_neighbors'], baris['mean_test_score'], marker='o')
plt.axvline(cari_knn.best_params_['model__n_neighbors'], color='red', linestyle='--',
            label=f"k terbaik = {cari_knn.best_params_['model__n_neighbors']}")
plt.xlabel('Nilai k')
plt.ylabel('f1_macro rata-rata 5 lipatan')
plt.title(f'Pengaruh nilai k terhadap kinerja k-NN ({best_w}, {best_m})')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## Langkah 4 — Melatih Naive Bayes

Menggunakan `GaussianNB` dengan penyetelan parameter `var_smoothing` pada rentang logaritmik hingga `1.0` untuk menstabilkan varians kolom biner hasil *One-Hot Encoding*.

In [ ]:
from sklearn.naive_bayes import GaussianNB

alur_nb = Pipeline([('prep', buat_preprocessor(pakai_skala=False)),
                    ('model', GaussianNB())])

ruang_nb = {'model__var_smoothing': np.logspace(-9, 0, 15)}

cari_nb = GridSearchCV(alur_nb, ruang_nb, cv=cv, scoring='f1_macro', n_jobs=-1)
cari_nb.fit(X_train, y_train)
nb_terbaik = cari_nb.best_estimator_

print('Parameter terbaik :', cari_nb.best_params_)
print('Skor f1_macro CV  :', round(cari_nb.best_score_, 4))

---
## Langkah 5 — Menyimpan artefak untuk Pertemuan 13

Seluruh hasil pertemuan ini disimpan dalam berkas `artefak_p12.joblib` di Google Drive untuk dievaluasi pada Pertemuan 13.

In [ ]:
artefak = {
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'numerik': numerik, 'kategorik': kategorik,
    'knn': knn_terbaik, 'nb': nb_terbaik,
    'param_knn': cari_knn.best_params_, 'param_nb': cari_nb.best_params_,
    'cv_knn': float(cari_knn.best_score_), 'cv_nb': float(cari_nb.best_score_),
    'kolom_encoding': int(contoh.shape[1]),
    'seed': SEED, 'versi_sklearn': sklearn.__version__,
}

jalur = os.path.join(FOLDER, 'artefak_p12.joblib')
joblib.dump(artefak, jalur)
print('Tersimpan:', jalur)
print('Ukuran   :', round(os.path.getsize(jalur) / 1024, 1), 'KB')